**Hybrid CNN-LSTM Model combines:**



*   CNN(Convolutional Neural Networks): Extracts spatial patterns from time series data

*   LSTM(Long Short-Term Memory): Captures temporal dependencies for movement analysis



In [18]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, LSTM , Dense , Dropout , Flatten , BatchNormalization
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.regularizers import l2
from tensorflow.keras import layers , models
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import confusion_matrix , classification_report
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

#warning ignore
import warnings
warnings.filterwarnings('ignore')
import keras #used to buid CNN

In [19]:
import random
import numpy as np
import tensorflow as tf

def set_reproducibility_seed(seed=42):
    # 1. Set the python built-in pseudo-random generator at a fixed value
    random.seed(seed)

    # 2. Set the numpy pseudo-random generator at a fixed value
    np.random.seed(seed)

    # 3. Set the tensorflow pseudo-random generator at a fixed value
    tf.random.set_seed(seed)

    # 4. Configure a new global session for TensorFlow (if using TF 1.x or legacy backends, otherwise tf.random.set_seed is sufficient)
    print(f"Global seed set to {seed}. Results should now be reproducible!")

set_reproducibility_seed(42)

Global seed set to 42. Results should now be reproducible!


**Load Dataset**

In [20]:
#load dataset
df = pd.read_csv('Clean_Dataset.csv')



In [21]:
df['Condition'].value_counts()

,count
Condition,
1,105886
0,29278


In [22]:
#encode categorical variables

label_encoders = {}
for col in ['Task']:
  label_encoders[col] = LabelEncoder()
  df[col] = label_encoders[col].fit_transform(df[col])


#map Device location to numerical values
df['Device location'] = df['Device location'].map({'RightWrist':0,'LeftWrist':1})

#define input feature (past condition + metadata) and target (predict condition)
features = ['Accel_X', 'Accel_Y', 'Accel_Z', 'Gyro_X', 'Gyro_Y', 'Gyro_Z', 'Task', 'Time', 'Device location', 'Age_At_Diagnosis', 'Gender', 'Handedness']
target = ['Condition']

**Create Data Sequence and Segmentation**

In [23]:
#sort data by patient_id and time
df = df.sort_values(by=['Patient_id'])

#Ensure strict sequencial train-test split (no shuffle and preserving order)

train_data_list = []
test_data_list = []

#Split and normlaize per patient to prevent data leakage
scaler = MinMaxScaler()

for patient_id, patient_df in df.groupby('Patient_id'):
  split_index = round(int(len(patient_df)*0.8))  #80% train data , 20% test data


  train_patient = patient_df.iloc[:split_index]
  test_patient = patient_df.iloc[split_index:]


  #Normalize per patient
  train_patient[features] = scaler.fit_transform(train_patient[features])
  test_patient[features] = scaler.transform(test_patient[features])

  train_data_list.append(train_patient)
  test_data_list.append(test_patient)

#Combine segmented data
train_data = pd.concat(train_data_list)
test_data = pd.concat(test_data_list)

#Function to create sequences for each patients data
sequence_length = 10
def create_sequences(data, sequence_length=sequence_length):
  sequences = []
  labels = []

  for patient_id , patient_df in data.groupby('Patient_id'):
    patient_data = patient_df.reset_index(drop=True)  #reset index for sequencial slicing


    for i in range(len(patient_data) - sequence_length):
      seq = patient_data.iloc[i:i+sequence_length][features].values
      label = patient_data.iloc[i+sequence_length][target].values  #predict next step

      sequences.append(seq)
      labels.append(label)

  return np.array(sequences) , np.array(labels)



#create sequences
x_train , y_train = create_sequences(train_data, sequence_length=sequence_length)
x_test , y_test = create_sequences(test_data, sequence_length = sequence_length)





**Define and Train Model**

In [26]:
# Define a standard L2 penalty value (e.g., 0.001 or 0.01)
l2_val = 0.001

In [27]:


from tensorflow.keras import regularizers

#Define hybrid CNN-LSTM model
model = Sequential([
    Conv1D(filters=64,kernel_size=3, activation='relu',input_shape=(x_train.shape[1],x_train.shape[2]),
           kernel_regularizer=regularizers.l2(l2_val)), # CNN feature extraction
    BatchNormalization(),  #Normalize activations
    MaxPooling1D(pool_size=2),
    Dropout(0.5),

    LSTM(64, return_sequences=True,
           kernel_regularizer=regularizers.l2(l2_val)), #LSTM for Temporal dependencies
    BatchNormalization(),
    Dropout(0.5),

    LSTM(32,return_sequences=False,
           kernel_regularizer=regularizers.l2(l2_val)), #second LSTM layer
    BatchNormalization(),
    Dropout(0.5),

    Dense(64, activation='relu',
           kernel_regularizer=regularizers.l2(l2_val)), #Dense layer after activation
    BatchNormalization(),
    Dropout(0.5),

    Dense(32,activation='relu',
           kernel_regularizer=regularizers.l2(l2_val)),
    Dropout(0.5),

    Dense(1,activation='sigmoid')  #output class for binary classification
])



In [ ]:
"""
#Define hybrid CNN-LSTM model
model = Sequential([
    Conv1D(filters=64,kernel_size=3, activation='relu',input_shape=(x_train.shape[1],x_train.shape[2])), # CNN feature extraction
    BatchNormalization(),  #Normalize activations
    MaxPooling1D(pool_size=2),
    Dropout(0.5),

    LSTM(64, return_sequences=True), #LSTM for Temporal dependencies
    BatchNormalization(),
    Dropout(0.5),

    LSTM(32,return_sequences=False), #second LSTM layer
    BatchNormalization(),
    Dropout(0.5),

    Dense(64, activation='relu'), #Dense layer after activation
    BatchNormalization(),
    Dropout(0.5),

    Dense(32,activation='relu'),
    Dropout(0.5),

    Dense(1,activation='sigmoid')  #output class for binary classification
])

"""


In [ ]:
#compile model
model.compile(optimizer='adam',loss='binary_crossentropy', metrics=['accuracy'])

#train model
history = model.fit(x_train, y_train, epochs=50, batch_size=32, validation_data=(x_test, y_test))


Epoch 1/50
3264/3264 ━━━━━━━━━━━━━━━━━━━━ 40s 10ms/step - accuracy: 0.8763 - loss: 0.3594 - val_accuracy: 0.8832 - val_loss: 0.3620
Epoch 2/50
3264/3264 ━━━━━━━━━━━━━━━━━━━━ 34s 10ms/step - accuracy: 0.8773 - loss: 0.3599 - val_accuracy: 0.8866 - val_loss: 0.3525
Epoch 3/50
3264/3264 ━━━━━━━━━━━━━━━━━━━━ 32s 10ms/step - accuracy: 0.8747 - loss: 0.3627 - val_accuracy: 0.8866 - val_loss: 0.3534
Epoch 4/50
3264/3264 ━━━━━━━━━━━━━━━━━━━━ 33s 10ms/step - accuracy: 0.8763 - loss: 0.3597 - val_accuracy: 0.8783 - val_loss: 0.3658
Epoch 5/50
3264/3264 ━━━━━━━━━━━━━━━━━━━━ 32s 10ms/step - accuracy: 0.8758 - loss: 0.3598 - val_accuracy: 0.8890 - val_loss: 0.3437
Epoch 6/50
3264/3264 ━━━━━━━━━━━━━━━━━━━━ 35s 11ms/step - accuracy: 0.8773 - loss: 0.3597 - val_accuracy: 0.8888 - val_loss: 0.3412
Epoch 7/50
3264/3264 ━━━━━━━━━━━━━━━━━━━━ 32s 10ms/step - accuracy: 0.8778 - loss: 0.3577 - val_accuracy: 0.8754 - val_loss: 0.3761
Epoch 8/50
3264/3264 ━━━━━━━━━━━━━━━━━━━━ 41s 10ms/step - accuracy: 0.8781 -

In [ ]:
# save Model
#model.save("../models/cnn_lstm_model2.keras")
print("Model training complete. Saved as 'cnn_lstm_model2.keras'.")

# evaluate Model performance
test_loss , test_acc = model.evaluate(x_test, y_test)
print(f'Test accuracy:{test_acc *100:.2f}%')
print(f'Test loss:{test_loss:.2f}')

In [ ]:
#Generate predictions
y_pred_prob = model.predict(x_test) #Get probabilities
y_pred = (y_pred_prob > 0.5).astype(int)  #convert to binary predictions

#Classification report
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred, target_names=["Healthy", "Parkinson's"]))



In [ ]:
# Extract loss & accuracy values

train_loss = history.history['loss']
val_loss = history.history['val_loss']
train_acc = history.history['accuracy']
val_acc = history.history['val_accuracy']

epochs = range(1, len(train_loss) + 1)

**Model Performance Visualization**

In [ ]:
# Plot Loss Curve
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(epochs, train_loss, 'b-', label='Training Loss')
plt.plot(epochs, val_loss, 'r-', label='Validation Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.title('Training & Validation Loss')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Plot Accuracy Curve
plt.subplot(1, 2, 2)
plt.plot(epochs, train_acc, 'b-', label='Training Accuracy')
plt.plot(epochs, val_acc, 'r-', label='Validation Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.title('Training & Validation Accuracy')
plt.legend()

plt.tight_layout()
plt.show()

**Save Model Performance for Tableau Visualization**

In [ ]:
# Save accuracy over epochs
accuracy_df = pd.DataFrame({
    'epoch': list(range(1, 51)),
    'train_accuracy': history.history['accuracy'],
    'val_accuracy': history.history['val_accuracy'],
    'train_loss': history.history['loss'],
    'val_loss': history.history['val_loss']
})
#accuracy_df.to_csv("accuracy_over_epochs.csv", index=False)

In [ ]:
# save classification report for Tableau visualization
y_pred_classes = np.argmax(y_pred, axis=1)
y_true_classes = np.argmax(y_test, axis=1)

report = classification_report(y_test, y_pred, target_names=["Healthy", "Parkinson's"], output_dict=True)
# Convert classification report to DataFrame
report_df = pd.DataFrame(report).transpose()
#report_df.to_csv("classification_report.csv", index=True)

# Save predictions along with true labels for Tableau visualization
results_df = pd.DataFrame({
    'True_Label': y_true_classes,
    'Predicted_Label': y_pred_classes,
    'Prediction_Confidence': np.max(y_pred, axis=1)  # Maximum probability score
})
#results_df.to_csv("model_predictions.csv", index=False)